### Links and Resources
| Concept | Reference |
|---|---|
| [Databricks AI Functions](https://docs.databricks.com/aws/en/large-language-models/ai-functions) | The whole catalog |
| [`ai_query`](https://docs.databricks.com/aws/en/sql/language-manual/functions/ai_query) | Generic LLM call |
| [`ai_classify`, `ai_extract`, `ai_translate`](https://docs.databricks.com/aws/en/large-language-models/ai-functions#task-specific-ai-functions) | Task-specific helpers |
| [`vector_search`](https://docs.databricks.com/aws/en/generative-ai/vector-search.html) | Embedding-based retrieval |
| [Foundation Model APIs](https://docs.databricks.com/aws/en/machine-learning/foundation-models/index.html) | Underlying endpoints |

In [ ]:
import os, sys

try:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.getOrCreate()
except ImportError:
    if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
        raise RuntimeError(
            f"Wrong Python kernel \u2014 'databricks.connect' not found in:\n  {sys.executable}\n\n"
            "In VSCode: click the kernel selector (top-right of notebook) "
            "and choose the interpreter at:\n"
            r"  C:\Source\Dsa\DataBricks\.venv\Scripts\python.exe"
        )
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()

from pyspark.sql import functions as F


def display(df, n=20):
    df.show(n, truncate=False)

# AI Functions

Databricks AI Functions are SQL-callable wrappers around foundation models hosted on Databricks. From a function-author's perspective they're "just functions" — you call them on a column, get back a column.

What makes them special:

- **No model orchestration code** — it's all SQL/PySpark.
- **Governed by Unity Catalog** — same `GRANT EXECUTE` model.
- **Pay-per-token** against a model serving endpoint, not per-cluster.
- **Vectorized** — Databricks batches rows under the hood.

This notebook is a tour. You'll need a workspace with **Foundation Model APIs enabled** to actually run the cells; the SQL reads identically across regions but the available models differ.

## Setup — confirm you have a model endpoint

By default, Databricks workspaces with FM APIs enabled have endpoints like `databricks-meta-llama-3-3-70b-instruct`, `databricks-claude-3-7-sonnet`, etc. Run the cell below to list what's available in **your** workspace; pick one for the examples that follow.

In [ ]:
# This requires `serving-endpoints` permission on your workspace.
# If it errors, ask your admin which FM endpoint name to use.
try:
    endpoints = spark.sql("SHOW PROVIDERS").collect()  # placeholder; real listing varies by workspace
except Exception as e:
    print("listing failed (expected if no permission):", e)

# Pick the endpoint string for ai_query examples.
ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"   # change to one available in your workspace
print("using endpoint:", ENDPOINT)

## A small demo dataset

In [ ]:
reviews = spark.createDataFrame([
    (1, "The shipping was fast and the laptop arrived in perfect condition. Five stars."),
    (2, "Awful customer support. Waited 40 minutes only to be disconnected."),
    (3, "It's fine. Does what it says. Battery could be better."),
    (4, "Le produit est arrivé endommagé et le service client n'a pas répondu."),
    (5, "Simply terrible. Would not buy again. Save your money."),
], ["id", "text"])
display(reviews)

## `ai_classify` — pick from a label list

You provide candidate labels; the model returns the best match. Cheap and consistent.

In [ ]:
reviews.createOrReplaceTempView("reviews")

# spark.sql("""
# SELECT id, text,
#        ai_classify(text, ARRAY('positive', 'neutral', 'negative')) AS sentiment
# FROM reviews
# """).show(truncate=False)

print("uncomment when running on a workspace with FM APIs enabled")

## `ai_extract` — pull structured fields

Pass a list of field names to extract; the model returns a struct.

In [ ]:
# spark.sql("""
# SELECT id,
#        ai_extract(text, ARRAY('product', 'issue', 'sentiment')) AS extracted
# FROM reviews
# """).select("id", "extracted.*").show(truncate=False)

print("uncomment when running on a workspace with FM APIs enabled")

## `ai_translate` — language → language

In [ ]:
# spark.sql("""
# SELECT id, text, ai_translate(text, 'en') AS text_en
# FROM reviews
# """).show(truncate=False)
print("uncomment when running on a workspace with FM APIs enabled")

## `ai_summarize` — TL;DR in N tokens

In [ ]:
# spark.sql("""
# SELECT ai_summarize(
#     concat_ws(' ', collect_list(text)),
#     50    -- max tokens
# ) AS overall_summary
# FROM reviews
# """).show(truncate=False)
print("uncomment when running on a workspace with FM APIs enabled")

## `ai_query` — generic prompt-on-a-column

The escape hatch for anything the task-specific functions don't cover. You write the prompt; the model answers per row.

```sql
ai_query(
    'endpoint-name',
    'prompt template referring to {value}',
    [optional schema for structured output]
)
```

Pattern: build a prompt string with `concat`/`format_string`, pass it to `ai_query`, optionally constrain the output with a `responseFormat` so you don't have to parse free text.

In [ ]:
# spark.sql(f"""
# SELECT id,
#        ai_query(
#            '{ENDPOINT}',
#            CONCAT(
#                'Rate this review on a 1-5 scale and explain in one short sentence. Review: ',
#                text
#            )
#        ) AS rating
# FROM reviews
# """).show(truncate=False)
print("uncomment when running on a workspace with FM APIs enabled")

### Structured output with `ai_query` — return a struct, not a string

A free-text LLM response is a parsing nightmare in production. Pass `responseFormat` (JSON schema) and the function returns a typed struct.

In [ ]:
# spark.sql(f"""
# SELECT id,
#        ai_query(
#            '{ENDPOINT}',
#            CONCAT('Analyze this review: ', text),
#            responseFormat => 'STRUCT<sentiment: STRING, rating: INT, language: STRING>'
#        ) AS analysis
# FROM reviews
# """).select("id", "analysis.*").show(truncate=False)
print("uncomment when running on a workspace with FM APIs enabled")

## `ai_gen` — text generation (no input column required)

For pure prompt-based generation that doesn't depend on table data. Less commonly useful in a SQL pipeline, more for ad-hoc text creation.

## `vector_search` — embedding retrieval

Different shape: it queries a **Vector Search index** (a managed embedding store you build separately from a Delta table). Returns the top-k nearest neighbours.

```sql
SELECT *
FROM vector_search(
    index => 'main.default.product_descriptions_idx',
    query_text => 'wireless headphones with noise cancellation',
    num_results => 5
)
```

Setup outside this notebook: create the index from a source Delta table via the Vector Search UI or `databricks.vector_search` Python SDK. Once created, `vector_search` is just a table-valued function.

## Cost, latency, and batching

- AI Functions call a model serving endpoint — **per-token billing**, not per-cluster. Big tables × big prompts × big models = real money. Sample first.
- They are vectorized: Databricks batches rows server-side, so 1000 rows isn't 1000 API calls. But the latency is still much higher than a built-in.
- Cache aggressively: write the AI Function output to a Delta table and join against it instead of re-calling per query.
- For high-throughput cases, consider **Foundation Model batch inference** APIs over `ai_query`.

## Determinism and reproducibility

LLM responses are **non-deterministic by default**. Two consequences:

- The same row recomputed tomorrow may give a different label. This breaks naive idempotence assumptions.
- Pin a model version when you can (`databricks-meta-llama-3-3-70b-instruct@version`), and persist outputs to a Delta table so the *result* is the source of truth, not the function call.

## Permissions and governance

Like every UC function: `GRANT EXECUTE ON FUNCTION ai_query TO ...`, plus `USAGE` on the parent schema/catalog. The model serving endpoint has its own ACL on top — having `EXECUTE` on `ai_query` doesn't automatically let you call any endpoint.

## Takeaways

- AI Functions turn LLMs into vectorized SQL functions. The interface is the same as a SQL UDF.
- **Task-specific** functions (`ai_classify`, `ai_extract`, `ai_translate`, `ai_summarize`) are cheaper and more consistent than generic prompts.
- Use **`ai_query` with `responseFormat`** for typed output — never parse free text in production.
- Persist results to Delta. LLM calls are slow, expensive, and non-deterministic; treat them like a costly remote API.
- `vector_search` is a separate beast — a table-valued function over a managed embedding index.
- This is the *application layer* of the function world: governed, scalable, but pay-per-token. Reach for it last, after built-ins, SQL UDFs, and Pandas UDFs have failed to express the task.